# Model Optimization: Pruning

In this notebook, we'll apply pruning techniques to our models using distributed processing. Instead of running the pruning on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the pruning on more powerful instances.

## What is Pruning?

Pruning is a technique that removes unnecessary weights from a neural network, effectively making the model more sparse. Research has shown that many neural networks are overparameterized, and a significant percentage of weights can be removed without substantial impact on accuracy.

### Benefits of Pruning:
- **Reduced Model Size**: Fewer parameters means smaller models
- **Faster Inference**: Fewer computations lead to faster inference
- **Lower Memory Requirements**: Sparse models require less memory
- **Reduced Overfitting**: Removing redundant weights can improve generalization

### Types of Pruning We'll Explore:
- **Unstructured Pruning**: Removes individual weights based on their magnitude (L1 norm)
- **Structured Pruning**: Removes entire structures like neurons or channels
- **Magnitude-based Pruning**: Removes weights with the smallest absolute values

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform pruning on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import os
import json
import torch
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput, Processor
from sagemaker.pytorch.processing import PyTorchProcessor

# Import our utility functions for distributed processing
from sagemaker_processing import run_pruning_job

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN
    %store OPTIMIZATION_INSTANCE_TYPE

## 3. Load Baseline Metrics and Model Information

In [ ]:
# Load baseline metrics from file
with open('baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

print(f"Loaded baseline metrics for {len(baseline_metrics)} models")

# Load model information from file
with open('model_info.json', 'r') as f:
    model_info = json.load(f)

print(f"Loaded information for {len(model_info)} models")

## 4. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment_analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question_answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked_lm": "The [MASK] is a large language model trained by OpenAI."
}

## 5. Create and Upload Pruning Script to S3

In this section, we'll create a Python script that performs the actual pruning. This script will be executed on the SageMaker Processing instances.

### What the Script Does:
1. **Loads the model and tokenizer** from Hugging Face
2. **Prepares sample inputs** for inference
3. **Applies pruning** using the specified method and amount
4. **Measures performance metrics** like model size and inference time
5. **Saves the pruned model** and metrics to the output directory

### Pruning Methods:
- **L1 Unstructured Pruning**: Removes individual weights with the smallest L1 norm (absolute value)
- **Random Unstructured Pruning**: Randomly removes weights regardless of their values
- **L2 Structured Pruning**: Removes entire structures (like neurons) based on their L2 norm

The pruning amount parameter controls what percentage of weights to remove. For example, a value of 0.3 means 30% of weights will be pruned.

In [ ]:
# Create a pruning script
pruning_script = """
import os
import json
import torch
import argparse
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoModelForTokenClassification, AutoModelForQuestionAnswering
from transformers import AutoModelForMaskedLM
from torch.nn.utils import prune

def load_model_and_tokenizer(model_name, task):
    """Load model and tokenizer based on task."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if task == "sequence-classification":
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
    elif task == "token-classification":
        model = AutoModelForTokenClassification.from_pretrained(model_name)
    elif task == "question-answering":
        model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    elif task == "masked-lm":
        model = AutoModelForMaskedLM.from_pretrained(model_name)
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    return model, tokenizer

def prepare_inputs(task, tokenizer, sample_input):
    """Prepare inputs for different model tasks."""
    if task == "sequence-classification":
        inputs = tokenizer(sample_input, return_tensors="pt")
    elif task == "token-classification":
        inputs = tokenizer(sample_input, return_tensors="pt")
    elif task == "question-answering":
        inputs = tokenizer(
            sample_input["question"],
            sample_input["context"],
            return_tensors="pt"
        )
    elif task == "masked-lm":
        inputs = tokenizer(sample_input, return_tensors="pt")
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    return inputs

def measure_inference_time(model, inputs, num_runs=10):
    """Measure inference time for a model."""
    # Warm-up run
    with torch.no_grad():
        _ = model(**inputs)
    
    # Measure inference time
    start_time = torch.cuda.Event(enable_timing=True)
    end_time = torch.cuda.Event(enable_timing=True)
    
    timings = []
    with torch.no_grad():
        for _ in range(num_runs):
            start_time.record()
            _ = model(**inputs)
            end_time.record()
            torch.cuda.synchronize()
            timings.append(start_time.elapsed_time(end_time))
    
    return sum(timings) / len(timings)

def get_model_size(model):
    """Get model size in MB."""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    size_mb = (param_size + buffer_size) / 1024**2
    return size_mb

    """Measure memory usage of a model."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        model.to('cuda')
        model.to('cpu')
        return memory_usage
    else:
        return get_model_size(model)

def apply_pruning(model, pruning_method="l1_unstructured", amount=0.3):
    """Apply pruning to a model."""
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            if pruning_method == "l1_unstructured":
                prune.l1_unstructured(module, name='weight', amount=amount)
            elif pruning_method == "random_unstructured":
                prune.random_unstructured(module, name='weight', amount=amount)
            elif pruning_method == "ln_structured":
                prune.ln_structured(module, name='weight', amount=amount, n=2, dim=0)
            
            # Make pruning permanent
            prune.remove(module, 'weight')
    
    return model

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model-info-path', type=str, required=True)
    parser.add_argument('--output-dir', type=str, required=True)
    parser.add_argument('--pruning-method', type=str, default='l1_unstructured')
    parser.add_argument('--pruning-amount', type=float, default=0.3)
    args = parser.parse_args()
    
    # Load model info
    with open(args.model_info_path, 'r') as f:
        model_info = json.load(f)
    
    # Process each model
    pruned_metrics = {}
    for model_key, info in model_info.items():
        print(f"Processing {model_key}: {info['model_name']}")
        
        # Load model and tokenizer
        model, tokenizer = load_model_and_tokenizer(info['model_name'], info['task'])
        
        # Define sample input
        if info['task'] == 'sequence-classification':
            sample_input = "This is a sample input for sentiment analysis."
        elif info['task'] == 'token-classification':
            sample_input = "John Smith works at Microsoft in Seattle."
        elif info['task'] == 'question-answering':
            sample_input = {
                "question": "What is machine learning?",
                "context": "Machine learning is a branch of artificial intelligence."
            }
        elif info['task'] == 'masked-lm':
            sample_input = "The [MASK] is a large language model."
        
        # Prepare inputs
        inputs = prepare_inputs(info['task'], tokenizer, sample_input)
        
        # Apply pruning
        pruned_model = apply_pruning(
            model, 
            pruning_method=args.pruning_method,
            amount=args.pruning_amount
        )
        
        # Move to GPU if available
        if torch.cuda.is_available():
            pruned_model = pruned_model.to('cuda')
            inputs = {k: v.to('cuda') for k, v in inputs.items()}
        
        # Measure metrics
        model_size = get_model_size(pruned_model)
        inference_time = measure_inference_time(pruned_model, inputs)
        num_parameters = sum(p.numel() for p in pruned_model.parameters())
        
        # Save pruned model
        output_dir = os.path.join(args.output_dir, model_key)
        os.makedirs(output_dir, exist_ok=True)
        pruned_model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        
        # Save metrics
        pruned_metrics[model_key] = {
            "model_key": model_key,
            "model_name": info['model_name'],
            "task": info['task'],
            "pruning_method": args.pruning_method,
            "pruning_amount": args.pruning_amount,
            "model_size": model_size,
            "inference_time": inference_time,
            "num_parameters": num_parameters
        }
    
    # Save metrics to file
    with open(os.path.join(args.output_dir, 'pruned_metrics.json'), 'w') as f:
        json.dump(pruned_metrics, f, indent=2)

if __name__ == '__main__':
    main()
"""

# Write the script to a file
with open('pruning_script.py', 'w') as f:
    f.write(pruning_script)

# Upload the pruning script to S3
s3_client = boto3.client('s3')
s3_client.upload_file(
    'pruning_script.py', 
    S3_BUCKET, 
    'scripts/pruning_script.py'
)

print(f"Created and uploaded pruning script to s3://{S3_BUCKET}/scripts/pruning_script.py")

## 6. Launch Distributed Pruning Jobs

Now we'll set up and launch the SageMaker Processing jobs to perform pruning. Each model will be processed in a separate job, allowing for parallel processing.

### Pruning Process:
1. **Create a PyTorch processor** with the appropriate instance type and configuration
2. **For each model**:
   - Save and upload model information to S3
   - Define inputs (pruning script and model info) and outputs
   - Launch a processing job with the appropriate arguments
   - Store the job information for monitoring

We're using L1 unstructured pruning with a pruning amount of 0.3 (30% of weights will be removed). This is a good starting point that typically provides significant size reduction with minimal impact on accuracy.

In [ ]:
# Define the instance type to use for pruning
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="1.13.1",
    py_version="py39",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="model-pruning",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch pruning jobs for each model
pruning_jobs = {}

for model_key in model_info.keys():
    print(f"\nLaunching pruning job for {model_key}...")
    
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/scripts/pruning_script.py',
            destination='/opt/ml/processing/input/code/pruning_script.py'
        ),
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data/model_info.json'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=f's3://{S3_BUCKET}/optimization/outputs/{model_key}'
        )
    ]
    
    # Run the processing job
    pruning_jobs[model_key] = processor.run(
        code='pruning_script.py',
        inputs=inputs,
        outputs=outputs,
        arguments=[
            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
            '--output-dir', '/opt/ml/processing/output',
            '--pruning-method', 'l1_unstructured',
            '--pruning-amount', '0.3'
        ]
    )
    
    print(f"Launched pruning job: {pruning_jobs[model_key].job_name}")

## 7. Monitor Job Status

After launching the pruning jobs, we need to monitor their progress. SageMaker Processing jobs run asynchronously, so we'll periodically check their status until all jobs are complete.

### Monitoring Process:
1. **Create a SageMaker client** to interact with the SageMaker API
2. **Check job status every 30 seconds** until all jobs are complete
3. **Display a status table** showing the current status of each job

Pruning is generally faster than training but can still take several minutes depending on the model size and instance type. The status table will update automatically to show the progress of each job.

In [ ]:
# Monitor job status
import time

# Create a SageMaker client
sagemaker_client = boto3.client('sagemaker')

# Check job status every 30 seconds
all_completed = False
while not all_completed:
    all_completed = True
    job_statuses = {}
    
    for model_key, job in pruning_jobs.items():
        response = sagemaker_client.describe_processing_job(
            ProcessingJobName=job.job_name
        )
        status = response['ProcessingJobStatus']
        job_statuses[model_key] = status
        
        if status in ['InProgress', 'Stopping']:
            all_completed = False
    
    # Display status table
    status_df = pd.DataFrame({
        'Model': list(job_statuses.keys()),
        'Status': list(job_statuses.values())
    })
    display(status_df)
    
    if not all_completed:
        print("Waiting for jobs to complete...")
        time.sleep(30)
    else:
        print("All jobs completed!")

## 8. Collect Results

Once all jobs are complete, we'll collect and combine the results from each job. Each job produces a metrics file containing information about the pruned model, such as size, inference time, and pruning parameters.

### Collection Process:
1. **Download metrics files** from S3 for each model
2. **Combine metrics** into a single dictionary
3. **Save combined metrics** to a local file for use in later notebooks

This gives us a comprehensive view of the pruning results across all models, which we'll analyze in the next section.

In [ ]:
# Download and combine results
pruned_metrics = {}

for model_key in model_info.keys():
    # Download metrics file
    try:
        s3_client.download_file(
            S3_BUCKET,
            f'optimization/outputs/{model_key}/pruned_metrics.json',
            f'temp_{model_key}_pruned_metrics.json'
        )
        
        # Load metrics
        with open(f'temp_{model_key}_pruned_metrics.json', 'r') as f:
            metrics = json.load(f)
        
        # Add to combined metrics
        pruned_metrics.update(metrics)
        
        print(f"Downloaded metrics for {model_key}")
    except Exception as e:
        print(f"Error downloading metrics for {model_key}: {e}")

# Save combined metrics
with open('pruned_metrics.json', 'w') as f:
    json.dump(pruned_metrics, f, indent=2)

print(f"\nSaved pruned metrics for {len(pruned_metrics)} models to pruned_metrics.json")

## 9. Compare Results

Now we'll compare the performance of the pruned models against the baseline models. This comparison helps us understand the impact of pruning on model size and inference speed.

### Key Metrics to Compare:
- **Model Size**: How much smaller are the pruned models?
- **Inference Time**: How much faster are the pruned models?
- **Size Reduction Percentage**: The percentage reduction in model size
- **Inference Speedup Percentage**: The percentage improvement in inference speed

We expect to see significant size reductions with minimal impact on inference time. In some cases, pruning can even improve inference speed due to the reduced computational load.

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

for model_key in pruned_metrics.keys():
    if model_key in baseline_metrics:
        baseline = baseline_metrics[model_key]
        pruned = pruned_metrics[model_key]
        
        # Calculate improvements
        size_reduction = (baseline['model_size'] - pruned['model_size']) / baseline['model_size'] * 100
        time_reduction = (baseline['inference_time'] - pruned['inference_time']) / baseline['inference_time'] * 100
        
        comparison_data.append({
            'Model': pruned['model_name'],
            'Pruning Method': pruned['pruning_method'],
            'Pruning Amount': f"{pruned['pruning_amount'] * 100:.1f}%",
            'Baseline Size (MB)': baseline['model_size'],
            'Pruned Size (MB)': pruned['model_size'],
            'Size Reduction (%)': size_reduction,
            'Baseline Inference (ms)': baseline['inference_time'],
            'Pruned Inference (ms)': pruned['inference_time'],
            'Inference Speedup (%)': time_reduction,
            'Baseline Memory (MB)': baseline['memory_usage'],
            'Pruned Memory (MB)': pruned['memory_usage'],
            'Memory Reduction (%)': memory_reduction
        })

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df

## 10. Next Steps

Now that we've applied pruning to our models, we'll explore knowledge distillation in the next notebook to create even smaller, faster models.

### What We've Learned:
- How to apply pruning to transformer models
- How to use SageMaker Processing for distributed optimization tasks
- How pruning affects model size and inference speed

### What's Next - Knowledge Distillation:
Knowledge distillation is a technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model. This allows us to create models that are much smaller and faster while retaining most of the accuracy of the original model. The combination of pruning and knowledge distillation can lead to even greater size reductions and performance improvements.